<img src="../logo.png" alt="Vegeta" width="240">

# GAVILAN — a sparrowhawk that guards a site against quadcopters (38)

*Gavilán* is the sparrowhawk in Spanish: *Accipiter nisus*, the ambush hunter that waits in cover, bursts out, makes a
short, hard pursuit and takes small birds in the air. That is what this machine became. It did not start there: the
brief asked for a goshawk that chases bandits through an industrial site, and the story below is how the design moved —
through eight layouts and five ways of catching — to a small X-wing that **waits on a charging perch as a camera**,
launches when a bandit comes near, and **throws a net from 3 to 5 m**.

| part | what | where the code is |
|---|---|---|
| 1 | the job, and what the lab's aircraft do at it (the gap) | `designs/gavilan_flight.py` (`fleet`, `kpi_table`) |
| 2 | the layouts tried: A, B, F, H, J, X, XP — and why each went | `designs/gavilan.py` |
| 3 | how to catch: ball gun, gas nets, talons, feet, slingshot; catching alive | `designs/gavilan_flight.py`, `designs/gavilan_slingshot.py`, `exp/gavilan_*` |
| 4 | the chosen Gavilan: airframe, slingshot, perch, mass | `designs/gavilan.py`, `gavilan_slingshot.py`, `gavilan_perch.py` |
| 5 | its numbers: the flight card the game reads | `gavilan_flight.flight_card` |
| 6 | the attack from the perch | `gavilan_flight.sling_attack` |
| 7 | what is left open | — |

**How to read the numbers.** Everything flies on reduced models — a point mass with a body axis, thrust along it, a
wing's polar blended into a flat plate past the stall (`gavilan_flight.Machine`) — on the drive, catalogue and drag
rules of ÉMERILLON (notebook 37) and the vortex lattice of Ogma. Hand estimates are marked where they enter (masses of
parts nobody has chosen, spring energies, standby power) and carry a 25 % margin. **No CFD, FEA or flight test stands
behind any number here**; the CFD cases are prepared (`sims/gavilan_cfd.py`), not run.

In [ ]:
# DEBUG — ON BY DEFAULT: the expensive solvers are mocked (vegeta.mock). Nothing in this notebook needs them.
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks

In [ ]:
import json, math, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
warnings.filterwarnings("ignore")
import gavilan as gv
import gavilan_flight as gf
import gavilan_slingshot as gs
import gavilan_perch as gpe
import gavilan_drawings as gd

DATA = Path("../designs/data")
FIG = DATA / "figures"
EXP = Path("../exp")
OUT = Path("../output") / "38_gavilan"; OUT.mkdir(parents=True, exist_ok=True)
RECOMPUTE = os.environ.get("GAVILAN_RECOMPUTE") == "1"     # the fleet tables take minutes: read from data/ unless asked
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:.2f}")

def show(name, width=1100):
    p = FIG / name
    display(Image(filename=str(p), width=width) if p.exists() else Markdown(f"*{name}: not drawn yet (`gavilan_drawings`)*"))

# Part 1 — the job, and the gap

**The brief.** Hostile quadcopters ("bandits") fly over an industrial site from 3 m between buildings to 150 m and
above, at 5 to 20 m/s; they stop, hover, reverse and drop behind walls. The hunter has to catch up and hold the bandit
in its camera at close range — and, as the work went on, catch it. Manoeuvre first; speed and endurance second.

**The lab's aircraft at that job.** MERLIN (a fast fixed-wing sampler), PEREGRINE (the folding-wing falcon), the
Kestrel X-wing and ÉMERILLON (the bullet quad) — each from its own module, unarmed, on the same manoeuvre models.

In [ ]:
kpi = json.loads((DATA / "gavilan_kpi.json").read_text())
if RECOMPUTE:
    kpi["rows"] = gf.kpi_table()
rows = {r["key"]: r for r in kpi["rows"]}
labels = {k["key"]: k["label"] for k in kpi["kpis"]}
lab = ["merlin", "peregrine", "xwing", "emerillon"]
gap = pd.DataFrame({k: {labels[c]: rows[k][c] for c in labels} for k in lab})
display(gap)
show("gavilan_fleet_turns.png")

**The gap.** The fixed wings turn hard only when fast (about 110 deg/s at 25 m/s), cannot fly below 11–12 m/s and
cannot stop: a bandit that hovers is overflown, and the next pass comes 9 s later. ÉMERILLON stops and turns, but on
6 minutes of cruise it cannot wait. The Kestrel X-wing hovers and turns but tops out at 30 m/s. Nothing in the lab
both stops beside a bandit **and** stays in the air long enough to be there when one comes.

# Part 2 — the layouts

Each was drawn in CadQuery (`designs/gavilan.py`) and flown on the same models. In order:

| layout | what | why it went (or stayed) |
|---|---|---|
| tilt-rotor | a wing with rotors that tilt | rejected before drawing: too expensive |
| **A** | tailsitter: short broad wing, four fixed motors on posts, long tail on four fins | the best chaser of the lab (score 0.94) — the base of what followed |
| **B** | A with the "T-fighter" side panels | same turning as A, half its cruise (lift spread over a short span) |
| **F** | plain fixed wing, two tractors, small wing | cannot stop; 8 s between passes; turns 32 deg/s at 15 m/s with a gun |
| **H** | F on a broad wing with an all-moving tail in the propellers' wash: flies past its stall like a hawk | 0.68: stops in 36 m, not 14 — it cannot point its thrust back |
| **J** | A cut for an ambush: one pack, guards, talons (a net on four legs) | the hawk's hunt: perch → sprint → flare → grab |
| **X** | J's motors and pack on four identical swept panels that are also its feet | lighter, narrower, fewer parts, no slower than J: **chosen** |
| **XP** | X with two hawk's feet (four-toed grippers) | ±6 cm of aiming against the net's ±22 cm: dropped once the net was thrown |

In [ ]:
for f in ("gavilan_3d.png", "gavilan_b_3d.png", "gavilan_f_3d.png", "gavilan_h_3d.png", "gavilan_j_3d.png", "gavilan_x_3d.png", "gavilan_xp_3d.png"):
    show(f, 900)

**Unarmed, on the chase's KPIs** (nine against bandits, six general; each scaled to the fleet's best, equal weights —
a summary to read beside the bars, not a verdict):

In [ ]:
sc = pd.DataFrame(kpi["scores"]).loc[["bandit", "general"]]
display(sc)
show("gavilan_kpi_bandit.png"); show("gavilan_kpi_scores.png")
show("gavilan_j_x_catchers.png")

# Part 3 — how to catch a bandit

| catcher | what | verdict |
|---|---|---|
| 8 mm steel balls | the recoilless emitter cut to a 0.30 m tube: 130 m/s, 18 J, 15 balls, 0.6 kg | works to 30 m — but it breaks the bandit |
| gas net launcher | the golf cannon with a 2.4 m net: 4–10 m | ~1 kg aboard, one shot per tank |
| light one-shot gas net | a 1.5 m net at 45 m/s from a burst diaphragm: 3–8 m, 5/5 | 0.69 kg in a pod |
| talons (a net on legs) | pushed onto the bandit at 3 m/s | ±22 cm, but the hunter must touch it |
| hawk's feet | two legs, four toes, a pincer | ±6 cm and 1.5 m/s: too precise |
| **latex slingshot** | the light net thrown by rubber bands, opposed: one net forward, one aft, no recoil | **chosen**: 0.46 kg at 30 m/s |

**Catching it alive.** Every net stops the rotors; nothing stops the fall. In the launcher's MuJoCo model a fouled
1.2 kg quadcopter hits the ground from 20 m at 16.6 m/s whatever the mesh; even a net spread flat above it only brings
that to 14.2 m/s — a 5 m/s landing needs about 0.8 m² of drag, a 1 m parachute. On an 8 m line with a stiff elastic
section the hunter holds it, but takes a 133 N jerk. **The decision: the lightest net now; the parachute (or a line)
later.**

In [ ]:
fall = json.loads((EXP / "gavilan_net_fall_results.json").read_text())
print("free fall after the catch (net_drone, 20 m up, shot from 5 m):")
for k, v in fall.items():
    downs = [s for s in v["shots"] if s.get("result") == "down"]
    if downs:
        print(f"  {k}: {len(downs)} fouled and down, impact {np.mean([s['impact_m_s'] for s in downs]):.1f} m/s")
# the net's best case, by hand: spread flat above the bandit, every cord broadside (Cd 1.2, 1 mm cord)
m, g, rho, h = 1.2 + 0.062, 9.81, 1.204, 20.0
for tag, cda in (("bandit alone (tumbling, 0.04 m2, hand)", 0.04), ("light net spread above it", 0.04 + 1.2 * 2 * 11 * 1.5 * 0.001)):
    k = 0.5 * rho * cda / m
    print(f"  {tag}: impact from {h:.0f} m {math.sqrt(g / k) * math.sqrt(1 - math.exp(-2 * k * h)):.1f} m/s")
print(f"  a 5 m/s landing needs {2 * m * g / (rho * 25):.2f} m2 of drag: a parachute {math.sqrt(4 * 2 * m * g / (rho * 25) / math.pi):.2f} m across")

**How fast must the net be thrown?** The light net opens to the same 1.75–1.95 m at 3–4 m whether it leaves at 15 or
45 m/s — the splay of its weights sets the opening; the speed sets the flight time and the drop. Thrown from a fixed
launcher at a quadcopter 4 m up (four shots a case):

In [ ]:
slow = json.loads((EXP / "gavilan_slow_net_results.json").read_text())
t = pd.DataFrame([{"net speed": k.split(",")[0], "bandit": k.split(",")[1].strip(), "range": k.split(",")[2].strip(), "captured": v}
                  for k, v in slow.items()])
display(t.pivot_table(index=["net speed", "bandit"], columns="range", values="captured", aggfunc="first"))

**30 m/s** is the slowest throw that catches a moving bandit as well as a hovering one, out to 5 m. Thrown from a
hunter that closes on the bandit, the closing speed adds to it.

**What throws it.** A spring launcher's weight is the energy, the spring's own mass (a spring spends 1/6 of its mass
× v² moving itself, so past a material's limit no amount of it throws faster) and the hardware. Latex stores about
ten times what spring steel does per gram; a steel coil cannot throw at 45 m/s at all:

In [ ]:
sl = pd.DataFrame(gs.table())
sl = sl[sl["speed_m_s"].isin([20.0, 30.0, 45.0])]
display(sl[["speed_m_s", "spring", "opposed", "energy_J", "spring_kg", "launcher_kg", "hunter_kick_m_s"]].set_index(["speed_m_s", "spring", "opposed"]))
show("gavilan_slingshot.png")

# Part 4 — the chosen Gavilan

**The concept.** A sparrowhawk guarding a site. It sits on a **perch** on the building, motors off, nose up, as a
camera — the tilting camera on the horizon, and on the sky above, where a quadcopter is blind. When a bandit comes
near it launches (0.3 s to spin up), makes **one short pursuit** of a few seconds, matches the bandit, and throws the
net from 3–5 m — level, or from above, where the net lands on the rotors and the hunter cannot fly into its own net.
Then it flies home and lands on its four feet in the perch's cups, where it charges.

**The airframe** is GAVILAN-X: a body (camera, one 6S pack, the stack), four identical swept panels at 45°, each with
a motor pod and a foot pad, guard rings on the propellers; the slingshot hangs under the belly.

In [ ]:
P = gv.x_resolve(gv.X_SLING)
geo = gv.GavilanX().generate(**gv.X_SLING)
print(f"valid {geo.shape.isValid()}, {len(geo.shape.Solids())} solids, box {np.round(geo.dimensions, 0)} mm")
geo.export_step(str(OUT / "gavilan.step"))
L = gv.GavilanX.layout(P); S = gv.GavilanX.sling_layout(P)
print({k: round(L[k], 1) for k in ("width", "foot_half_width", "tip_to_body", "tip_to_tip", "S_mm2")},
      {k: round(S[k], 1) for k in ("length", "clear_of_lower_discs", "clear_of_camera")})
mb = gf.gavilan_mass_budget()
display(pd.DataFrame([{"item": i.name, "g": round(i.kg * 1e3, 1), "group": i.group, "source": i.source} for i in mb.items]).set_index("item"))
print(f"all-up {mb.mass:.3f} kg;", {k: round(v, 3) for k, v in mb.by_group().items()})

**The perch.** A cross of four arms; under each foot a cone that funnels the pad in (±46 mm, ±8° of yaw) onto a
sprung brass contact. An ideal-diode bridge aboard makes any foot right in any cup; a small balancer aboard; the perch
gives a 1C constant-current, constant-voltage charge. On the perch it watches for ever; off it, on its pack.

In [ ]:
display(pd.Series(gpe.budget(), name="the perch's budget").to_frame())
show("gavilan_perch.png")

# Part 5 — the flight card

What a game (Armageddon) reads, without Vegeta: `designs/data/gavilan_flight_card.json`, written here from the models.
Turn rates are instantaneous (the speed is spent); the sustained rate holds the speed.

In [ ]:
card = gf.flight_card()
(DATA / "gavilan_flight_card.json").write_text(json.dumps(card, indent=1, ensure_ascii=False))
rec = gf.design_record()
(DATA / "gavilan_design.json").write_text(json.dumps(rec, indent=1, ensure_ascii=False, default=str))
flat = {k: v for k, v in card.items() if not isinstance(v, (dict, list))}
display(pd.Series(flat, name="Gavilan").to_frame())
display(pd.DataFrame(card["turns"]).set_index("speed_m_s"))
display(pd.Series(card["endurance_min"], name="endurance [min] / watch [h]").to_frame())

# Part 6 — the attack from the perch

The hunter sprints, flares to the fastest closing speed that still lets it stop 1.5 m short of the bandit after the
throw (`fastest_safe_shot`), throws from 4 m, and the net flies at the slingshot's 30 m/s plus that closing speed.

In [ ]:
att = pd.DataFrame([gf.sling_attack(d) for d in (50.0, 100.0, 200.0)]).set_index("distance_m")
display(att)
show("gavilan_hawk.png")

**From above.** A quadcopter's rotors face up and its camera looks forward and down. Hanging above it, nose up,
Gavilan's slingshot points its **rear** net straight down: the counter-mass of the opposed slingshot is a second net
with a use. Fired down, gravity closes the net instead of making it drop short, and the bandit falls away from the
hunter. Not yet simulated (Part 7).

# Part 7 — what is left open

- **The catch from a moving hunter**, level and from above: the capture tests are from a fixed launcher. A downward
  shot in the launcher's model is the next run.
- **The ambush as one simulation**: a bandit approaching a building, the launch, the pursuit among boxes, the throw,
  the return to the perch — and Armageddon flying it.
- **After the catch**: a parachute or a line. A line needs an elastic section soft enough to keep the jerk under the
  hunter's spare thrust (the stiff one gave 133 N; about 66 N are spare).
- **The airframe's aerodynamics past the stall**: the CFD sweeps are prepared (`sims/gavilan_cfd.py`), not run; no
  FEA of the panels or the 8 g load factor.
- **The slingshot**: built and chronographed — the latex energy, the release, the cocked bands over hours on the perch.
- **The exchange file** in `exports/` (a Chiron robot for `tools/publish.py`), so Montezuma and Armageddon draw the
  real shape.
- **Hand estimates** to replace with chosen parts: the slingshot, the perch's charging parts, standby power, guards.